####  감성분석 -> 머신러닝
데이터셋 : 전처리
- BoW 모델
    - 단어를 특성 벡터로 변환
    - tf-idf 단어 적합성 평가
    - 텍스트 데이터 정제
    - 문서를 토큰으로 나누기
- LogisticRegession


glob
    - 파이썬에서 파일 경로를 패턴(와일드카드)으로 검색할 때 사용하는 모듈
    - 특정 폴더 안에서 내가 원하는 형식의 파일만 찾아주는 역할을 합니다.
- 기본 개념
    - \* : 0개 이상의 임의의 문자
    - ? : 한 글자
    - [abc] : a, b, c 중 하나

데이터 사이트
http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

In [21]:
# 데이터 조회 : with, pd.DataFrame([]) , glob , concat
import pandas as pd
from glob import glob

# train_neg_df 생성
file_lists = glob("C:\\김지은\\데이터베이스\\aclImdb\\train\\neg\\*.txt")
pd_lists = []
for file_path in file_lists[:500]:
    with open(file_path,'r',encoding='utf-8') as f: 
        data ={
            'review' : f.read(),
            'target' : 0
        }
        df = pd.DataFrame([data])
        pd_lists.append(df)
train_neg_df = pd.concat(pd_lists,ignore_index=True)
train_neg_df.head()

# (x) : df = pd.DataFrame([f.read()]) # ([])
# 위 방법과 (x) 방법과의 차이

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0


In [22]:
# positive 동일하게 train_pos_df
# train_df = pd.concat([train_neg_df, train_pos_df])
# movie_data.csv

In [ ]:
# 강사풀이

In [23]:
# train_pos_df 생성
file_lists = glob("C:\\김지은\\데이터베이스\\aclImdb\\train\\pos\\*.txt")
pd_lists = []
for file_path in file_lists[:500]:
    with open(file_path,'r',encoding='utf-8') as f: 
        data ={
            'review' : f.read(),
            'target' : 0
        }
        df = pd.DataFrame([data])
        pd_lists.append(df)
train_pos_df = pd.concat(pd_lists,ignore_index=True)
train_pos_df.head()

,review,target
0,Bromwell High is a cartoon comedy. It ran at t...,0
1,Homelessness (or Houselessness as George Carli...,0
2,Brilliant over-acting by Lesley Ann Warren. Be...,0
3,This is easily the most underrated film inn th...,0
4,This is not the typical Mel Brooks film. It wa...,0


In [ ]:
# concat으로 train_neg_df, train_pos_df를 하나의 데이터 프레임 train_df로 합치기
train_df = pd.concat([train_neg_df, train_pos_df])
train_df.head()

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0


In [ ]:
# # train_df 를 movie_data.csv로 저장하기
train_df.to_csv('movie_data.csv',index=False,encoding='utf-8')

In [28]:
# 만든 csv 파일 불러오기
df = pd.read_csv('movie_data.csv')
df.head()

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0


In [ ]:
# BoW(Bag of Words) 모델
# 문자를 숫자벡터
# 단어의 등장횟수를 카운트

# 전체 훈련데이터에서 모든 고유한 단어(토큰)로 어휘 사전
# 각 문서(리뷰데이터)를 사전을 기준으로 벡터화  N번째단어가 문서에서 3번나오면 벡터의 N번째값이 3이 된다.
# 문서1 : "나는 영화가 좋다"
# 문서2 : "나는 영화가 싫다"
# 사전 : {'나는':0, '영화가':1,'좋다':2,'싫다':3}
# 벡터화는 사전의 크기만큼 모든 문장의 길이를 동일하게
# 문서1벡터 : [0,1,2] - > [1,1,1,0]
# 문서2벡터 : [0,1,3] - > [1,1,0,1]

In [ ]:
# BoW 모델 사용 예시
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
count= CountVectorizer() # One-Hot Encoder 와 유사
docs=([
    'The sun is shining',
    'The weather is sweet'
])
bag = count.fit_transform(docs)

In [36]:
count.vocabulary_

{'the': 4, 'sun': 2, 'is': 0, 'shining': 1, 'weather': 5, 'sweet': 3}

In [ ]:
# 문서 d 에 등장한  단어 t의 횟수를 tf(t,d)
# BoW를 보완하고, 좀더 정교한 텍스트 벡터화 방식 : TF-IDF(Term Frequency - Inverse Domcument Frequency)
# TF(t,d) : 단어 t 가 문장 D에 나타난 횟수 / 문서 d의 모든 단어 수 
# IDF(t,D) : log( 총 문서수 |D| /  단어 t를 포함한 문서의 수 df(t) ) -- log 단어의 희귀성을 너무 과하게 반영하지 않도록 스케일링
# 분모에 +1(사이킷 런의 경우) : 분모가 0 이 되는 것을 방지
    # 분모가 0인경우 발생하는 오류 : ZeroDivisionError
# log( 1+|D| / 1+df(f) )

# TF-IDF(t,d,D) = TF(t,d) x IDF(t,D
# 단어 중요도를 보여준다.

# "나는"
# TF : 리뷰에 3번 나옴 (높음)
# IDF : 전체 10,000개 리뷰중에 9000개 나옴(매우 낮음)
# TF-IDF = 높음 x 매우낮음 = 낮음(중요도가 낮음)

# "명작" TF : 리뷰에 2번 나옴 (높음)
# IDF : 전체 10,000개 리뷰중에 50개 나옴(매우 높음)
# TF-IDF =  높음 x 매우높음 = 높음(핵심단어)
# 적게 언급 된다.

In [ ]:
# 데이터 정제... html tag 와 같은 불필요한 string이 보임... 특수기호 기타 등등. < - '

# 데이터 정제하는 함수 생성
import re
def preprocessor(s):
    # 1. 영문, 공백, ., . 만 남기기
    clean = re.sub(r'[^A-Za-z\s.,]+','',s)
    # 2. 연속된 마침표(...)를 마침표 . 하나로
    clean = re.sub(r'\.{2.}', '.', clean)
    # 3. 연속된 공백 처리
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

In [42]:
df['review'] = df.review.apply(preprocessor)

In [46]:
# 문서를 토큰으로 나누기
%pip install nltk
# 콘다는 이런 설치 명령어를 입력한다.
# conda install -c anaconda nltk

Note: you may need to restart the kernel to use updated packages.


In [51]:
from nltk.stem.porter import PorterStemmer
# PorterStemmer()는 자연어 처리(NLP)에서 단어의 어간(stem)을 추출하는 도구
# 단어를 어간(stem)으로 변환하기 : PorterStemmer().stem("단어")
# 어간 :
    # 단어에서 의미를 유지하는 최소한의 형태
    # 접사(suffix, prefix)나 굴절(변형)을 제거한 기본 형태
    # 반드시 사전에 존재하는 단어일 필요는 없음
# 어간의 기준 :
    # 규칙 기반: -ing, -ed, -s, -ly 등 일반적인 접미사 제거
    # 최소 길이 유지: 너무 짧게 자르지 않음
    # 의미 유지 우선: 단어의 핵심 의미가 남도록 처리

# 토큰화하는 방법 1 : text.split()
def tokenizer(text):
    return  text.split()

# 토큰화하는 방법 2 : PorterStemmer()
porter = PorterStemmer()
def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

# text.split() 과 PorterStemmer() 의 차이 비교 ^
# text.split() → 단순히 문자열을 공백이나 구분자로 쪼개서 단어 리스트를 만든다.
# PorterStemmer() → 단어를 기본 형태(어간)로 바꿔서 의미를 일반화한다.
# 즉, split()은 단어를 나누는 것, PorterStemmer()는 단어를 줄여서 표준화하는 것이에요.

# split() → 단어를 분리만 함
# PorterStemmer() → 단어를 의미 단위로 일반화함

# 어간추출 Stemming : 단어의 접미사 -s -es -ing -ed 등등.. 를 강제로 제거해서 단어의 원형을 찾는과정
tokenizer(df.review[0][:100])

['Story',
 'of',
 'a',
 'man',
 'who',
 'has',
 'unnatural',
 'feelings',
 'for',
 'a',
 'pig.',
 'Starts',
 'out',
 'with',
 'a',
 'opening',
 'scene',
 'that',
 'is',
 'a',
 'terri']

In [50]:
tokenizer('runners like running')

['runners', 'like', 'running']

In [52]:
# 불용어
# 불용어(stop words)는 자연어 처리(NLP)에서 분석에 큰 의미를 주지 않는 단어
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords') # 불용어 사전 다운로드
stopwords.words('english')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [ ]:
# 독립변수 / 종속변수 생성
X =
y =

# 훈련 / 테스트 데이터 분할
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

SyntaxError: invalid syntax (1042026116.py, line 2)